# QMedViT Unified Notebook

Single end-to-end pipeline for classical and quantum MedViT variants. Pick a variant in the config cell below; the rest of the notebook is identical to the per-size classical notebooks (install → dataset → train → test → FGSM/PGD adversarial eval).

**Available `MODEL_VARIANT` options:**

| Variant | Train | Test | Description |
| --- | --- | --- | --- |
| `classical_small` | — | — | Baseline `MedViT.MedViT_small` |
| `classical_base` | — | — | Baseline `MedViT.MedViT_base` |
| `classical_large` | — | — | Baseline `MedViT.MedViT_large` |
| `quantum_softmax_gpu_gpu` | analytic sim | analytic sim | Trains and evaluates with the fast exact-statevector path (`qpu_mode=False`). Differentiable through the quantum softmax. |
| `quantum_softmax_gpu_qpu` | analytic sim | shots sim (`qpu_shots=5000`) | Trains analytically, then rebuilds the model in `qpu_mode=True` and copies weights for eval — mimics deploying a trained model on a noisy/shot-limited backend. |
| `quantum_softmax_qpu_qpu` | shots sim (`qpu_shots=5000`) | shots sim (`qpu_shots=5000`) | Trains and evaluates with finite-shot sampling. Quantum softmax weights are frozen during training (QPU path detaches gradients); the rest of the network still trains. Very slow. |
| `quantum_quanv_stem` | classical SGD | classical SGD | Replaces the first ConvBNReLU of the stem (3→64) with a Quanvolutional layer (Henderson et al. 2019, arXiv:1904.04767). 64 random 9-qubit circuits sampled once and frozen; outputs precomputed into a 512-entry lookup table per filter. The quantum part is non-trainable — only the BN/ReLU after the quanv layer and all classical layers downstream are trained. |
| `quantum_quanv_stem_gpu_qpu` | analytic table (init) | shots-sampled table (init, `qpu_shots=5000`) | Lookup table built via analytic statevector at training-model construction, but rebuilt with finite-shot sampling at eval-model construction. The forward path stays a lookup in both cases — QPU mode adds shot noise to the table VALUES, not the forward call. Random circuits are seeded identically so train/eval share the same projection up to sampling noise. |
| `quantum_quanv_hur2022_c2` | classical SGD + quantum backprop | classical SGD + quantum backprop | Replaces every ConvBNReLU.conv in the stem with a trainable parametrized quanvolutional layer based on Hur et al. 2022 (arXiv:2108.00661v2). Uses Hur Fig. 2b **Convolutional Circuit 2** (H + CNOT + 2 RX rotations) as a translationally-invariant brick-wall ansatz over $k^2=9$ qubits, with 2 angles per layer shared across all qubit pairs AND across input channels (Hur §IIB). Encoding is continuous $R_y(\pi (x+1)/2)$ so gradients flow back to the pixels. PennyLane `diff_method="backprop"`; ~256 quantum parameters for stem block 0. |
| `quantum_quanv_hur2022_c2_28_l2` | classical SGD + quantum backprop | classical SGD + quantum backprop | Opțiunea 2: native 28×28 MedMNIST input, only stem[0] (3→64) replaced by the Hur 2022 Circuit 2 quanvolutional layer with `n_ansatz_layers=2`. Other stem blocks remain classical. Drastically lowers per-epoch cost vs. the full-stem quantum variant. Architectural risk: at 28×28 input the transformer stages operate on very small feature maps. |
| `quantum_quanv_hur2022_c2_28_l1` | classical SGD + quantum backprop | classical SGD + quantum backprop | Opțiunea 2-lite: same as `_28_l2` but `n_ansatz_layers=1` (single ansatz layer → fewer quantum parameters and ~2× faster per-step quantum cost). Use this when `_28_l2` is still too slow or when investigating whether the deeper ansatz adds value. |
| `quantum_quanv_hur2022_c2_original_lgpu` | classical SGD + quantum backprop on GPU | classical SGD + quantum backprop on GPU | Original full-stem Hur 2022 Circuit 2 configuration (224×224 input, all four stem ConvBNReLU blocks quanvolutional, `n_ansatz_layers=2`) pinned to `lightning.gpu` for T4-level throughput. Auto-falls back to `lightning.qubit` (C++ CPU) or `default.qubit` if `lightning.gpu` cannot be loaded. `qnode_chunk_size=16384` to amortize per-call dispatch overhead. Realistic per-epoch wallclock on a T4: ~1-2.5 hours; on an A100: ~20-60 min. Use this for full-architecture quantum experiments where the `_28_l*` variants are too aggressive a compromise. |
| `quantum_quanv_hur2022_c9` | classical SGD + quantum backprop | classical SGD + quantum backprop | Same as `hur2022_c2` but using Hur Fig. 2i **Convolutional Circuit 9** (arbitrary $SU(4)$ block with three $U_3$ gates per pair, 15 angles per layer). Same translationally-invariant brick-wall extension and per-output-channel parameter sharing. More expressive but ~7.5× more quantum parameters; use this when Circuit 2 underfits. |
| `quantum_quanv_henderson2019` | classical SGD | classical SGD | Replaces every ConvBNReLU.conv in the stem with a Henderson 2019 quanvolutional layer (arXiv:1904.04767). Faithful to the original paper: threshold encoding (pixels > 0 get a `PauliX`), per-output-channel fixed random RX/RY/RZ + CNOT circuit (no `nn.Parameter`, no parameter-shift), analytic `<Z>`-expval decoding with a unique-bitstring lookup. Channel-wise (9 qubits per patch) by default for NISQ feasibility. The quantum block is non-trainable; BN/ReLU and all downstream layers train classically. |
| `quantum_quanv_all_stem` | classical SGD | classical SGD | All four stem ConvBNReLU blocks replaced. Block 0 uses QuanvStem (RGB-tuned, channel-mean reduction); blocks 1-3 use QuanvBlock (per-filter random sampling of 9 positions across the C*k*k unfolded patch). All four blocks are frozen random projections; only BN gamma/beta train. Total of 64+32+64+64=224 random 9-qubit circuits built once at construction. |

The "GPU" path is PennyLane's `default.qubit` analytic statevector simulator (no shots — the model is differentiable); the "QPU" path is the same device with `shots=5000`, mimicking what real hardware does (sampled probabilities, no gradients through the quantum block).

## Install Requirements

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/alexandrachirita98/MedViT-Quantum/

In [ ]:
%cd /kaggle/working/MedViT-Quantum

In [ ]:
%pwd

In [ ]:
pip install -r requirements.txt

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

import torchvision
import torchvision.utils
from torchvision import models
import torchvision.datasets as dsets
import torchvision.transforms as transforms
from torchsummary import summary

from tqdm import tqdm
import medmnist
from medmnist import INFO, Evaluator

import torchattacks
from torchattacks import PGD, FGSM

In [ ]:
print("PyTorch", torch.__version__)
print("Torchvision", torchvision.__version__)
print("Torchattacks", torchattacks.__version__)
print("Numpy", np.__version__)
print("Medmnist", medmnist.__version__)

## Configuration

Edit this cell to choose the model variant, dataset, and training hyperparameters. Everything below this cell is variant-agnostic.

In [ ]:
MODEL_VARIANT = "quantum_quanv_hur2022_c2_28_l1"

DATA_FLAG = "retinamnist"
# [tissuemnist, pathmnist, chestmnist, dermamnist, octmnist,
# pnemoniamnist, retinamnist, breastmnist, bloodmnist, tissuemnist,
# organamnist, organcmnist, organsmnist]

NUM_EPOCHS = 10
BATCH_SIZE = 10
LR = 0.005

QPU_SHOTS = 5000

from MedViT import MedViT_small, MedViT_base, MedViT_large
from quantum_variants.softmax_only import QMedViT_Softmax_Only
from quantum_variants.quanv_2d_henderson_2019 import QMedViTHenderson2019
from quantum_variants.quanv_2d_hur_2022 import QMedViTHur2022


def _qsoftmax(qpu, n):
    return QMedViT_Softmax_Only(
        stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
        num_classes=n, qpu_mode=qpu, qpu_shots=QPU_SHOTS,
    )


def _qhenderson2019(n, seed=0):
    return QMedViTHenderson2019(
        stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
        num_classes=n, quanv_seed=seed,
    )


def _qhur2022_c2(n):
    return QMedViTHur2022(
        stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
        num_classes=n, ansatz="circuit_2", n_ansatz_layers=2,
    )


def _qhur2022_c9(n):
    return QMedViTHur2022(
        stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
        num_classes=n, ansatz="circuit_9", n_ansatz_layers=2,
    )


def _qhur2022_c2_28_l2(n):
    # Note: lightning.qubit (C++ CPU) is used here because lightning.gpu
    # is unreliable on Kaggle (requires cuQuantum drivers). lightning.qubit
    # is always available once `pennylane-lightning` is installed.
    return QMedViTHur2022(
        stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
        num_classes=n,
        ansatz="circuit_2",
        n_ansatz_layers=2,
        quantum_layer_indices=[0],
        qdevice="lightning.qubit",
    )


def _qhur2022_c2_28_l1(n):
    # See `_qhur2022_c2_28_l2` for why lightning.qubit is preferred on Kaggle.
    return QMedViTHur2022(
        stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
        num_classes=n,
        ansatz="circuit_2",
        n_ansatz_layers=1,
        quantum_layer_indices=[0],
        qdevice="lightning.qubit",
    )


def _qhur2022_c2_original_lgpu(n):
    """Original Hur 2022 configuration (Hur §IIIA1 Circuit 2, full stem
    quantization, n_ansatz_layers=2) targeting an NVIDIA T4 (or better)
    via lightning.gpu. Automatically falls back to lightning.qubit /
    default.qubit if lightning.gpu is unavailable.
    """
    return QMedViTHur2022(
        stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
        num_classes=n,
        ansatz="circuit_2",
        n_ansatz_layers=2,
        quantum_layer_indices=None,   # full stem (all four ConvBNReLU)
        qdevice="lightning.gpu",
        qnode_chunk_size=16384,       # T4 has 16GB VRAM; larger chunks
                                      # amortize per-call overhead better
                                      # at the cost of more peak memory
    )


MODEL_BUILDERS = {
    "classical_small": {"train": lambda n: MedViT_small(num_classes=n), "eval_swap": None},
    "classical_base":  {"train": lambda n: MedViT_base(num_classes=n),  "eval_swap": None},
    "classical_large": {"train": lambda n: MedViT_large(num_classes=n), "eval_swap": None},
    "quantum_softmax_gpu_gpu": {
        "train": lambda n: _qsoftmax(False, n),
        "eval_swap": None,
    },
    "quantum_softmax_gpu_qpu": {
        "train": lambda n: _qsoftmax(False, n),
        "eval_swap": lambda n: _qsoftmax(True, n),
    },
    "quantum_softmax_qpu_qpu": {
        "train": lambda n: _qsoftmax(True, n),
        "eval_swap": None,
    },

    "quantum_quanv_henderson2019": {
        "train": lambda n: _qhenderson2019(n),
        "eval_swap": None,
    },
    "quantum_quanv_hur2022_c2": {
        "train": lambda n: _qhur2022_c2(n),
        "eval_swap": None,
    },
    "quantum_quanv_hur2022_c9": {
        "train": lambda n: _qhur2022_c9(n),
        "eval_swap": None,
    },
    "quantum_quanv_hur2022_c2_28_l2": {
        "train": lambda n: _qhur2022_c2_28_l2(n),
        "eval_swap": None,
        "input_size": 28,
    },
    "quantum_quanv_hur2022_c2_28_l1": {
        "train": lambda n: _qhur2022_c2_28_l1(n),
        "eval_swap": None,
        "input_size": 28,
    },
    "quantum_quanv_hur2022_c2_original_lgpu": {
        "train": lambda n: _qhur2022_c2_original_lgpu(n),
        "eval_swap": None,
    },
}

if MODEL_VARIANT not in MODEL_BUILDERS:
    raise ValueError(
        f"Unknown MODEL_VARIANT '{MODEL_VARIANT}'. "
        f"Expected one of: {list(MODEL_BUILDERS)}"
    )

print(f"Selected variant: {MODEL_VARIANT}")

## Dataset

In [ ]:
data_flag = DATA_FLAG
download = True

info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])

print("number of channels : ", n_channels)
print("number of classes : ", n_classes)

In [ ]:
INPUT_SIZE = MODEL_BUILDERS[MODEL_VARIANT].get("input_size", 224)

from torchvision.transforms.transforms import Resize
# preprocessing
train_transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.Lambda(lambda image: image.convert('RGB')),
    torchvision.transforms.AugMix(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])
test_transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.Lambda(lambda image: image.convert('RGB')),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# load the data
train_dataset = DataClass(split='train', transform=train_transform, download=download)
test_dataset = DataClass(split='test', transform=test_transform, download=download)

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
train_loader_at_eval = data.DataLoader(dataset=train_dataset, batch_size=2*BATCH_SIZE, shuffle=False)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*BATCH_SIZE, shuffle=False)

print(f"Dataset configured for INPUT_SIZE={INPUT_SIZE}")

In [ ]:
print(train_dataset)
print("===================")
print(test_dataset)

## Model

In [ ]:
model = MODEL_BUILDERS[MODEL_VARIANT]["train"](n_classes).cuda()

## Profile (estimate epoch time before training)

In [ ]:
# === Forward profiling: per-stem-layer breakdown + epoch estimate ===
# Runs BEFORE training so you can decide whether to abort and reduce the
# config. Times are reported per stem ConvBNReLU block; the heaviest layers
# are usually the prime targets for selective quantization.

import time
import torch

_was_training = model.training
model.eval()

# Use a SMALL batch first (size 1) to characterize per-image cost, then the
# full BATCH_SIZE. This separates fixed per-call overhead from per-patch cost.
def _profile_one(name, x):
    """Run one timed forward with per-stem-layer hooks. Returns dict of
    {layer_name: seconds}."""
    layer_times, layer_starts = {}, {}

    def make_pre(n):
        def hook(_m, _inp):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            layer_starts[n] = time.perf_counter()
        return hook

    def make_post(n):
        def hook(_m, _inp, _out):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            layer_times[n] = time.perf_counter() - layer_starts[n]
        return hook

    handles = []
    if hasattr(model, "stem"):
        for i, layer in enumerate(model.stem):
            handles.append(layer.register_forward_pre_hook(make_pre(f"stem[{i}]")))
            handles.append(layer.register_forward_hook(make_post(f"stem[{i}]")))

    # Warmup (don't time): first run pays JIT / CUDA allocator cost.
    with torch.no_grad():
        _ = model(x)

    # Timed run.
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(x)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    total = time.perf_counter() - t0

    for h in handles:
        h.remove()
    return total, layer_times

print(f"Profiling MODEL_VARIANT = {MODEL_VARIANT!r}\n")

# Try to discover image size from the first training tensor; fall back to 224.
try:
    _img_size = next(iter(train_loader))[0].shape[-1]
except Exception:
    _img_size = 224
print(f"Image size detected: {_img_size}x{_img_size}")

for batch in (1, BATCH_SIZE):
    x_prof = torch.randn(batch, 3, _img_size, _img_size).cuda()
    total, layer_times = _profile_one(f"b{batch}", x_prof)
    print(f"\n--- batch_size={batch} ---")
    print(f"Total forward: {total*1000:>8.1f} ms")
    if layer_times:
        stem_total = sum(layer_times.values())
        for name, t in sorted(layer_times.items()):
            pct = 100.0 * t / total if total > 0 else 0.0
            print(f"  {name:12s}: {t*1000:>8.1f} ms  ({pct:>4.1f}% of forward)")
        print(f"  {'stem total':12s}: {stem_total*1000:>8.1f} ms  "
              f"({100.0*stem_total/total:.1f}% of forward)")

# Epoch estimate using full BATCH_SIZE measurement.
try:
    n_train_batches = len(train_loader)
except Exception:
    n_train_batches = 108  # rough fallback for RetinaMNIST at batch=10

x_full = torch.randn(BATCH_SIZE, 3, _img_size, _img_size).cuda()
fwd_full, _ = _profile_one("full", x_full)
# Backward is typically 1.5-2.5x forward for hybrid quantum-classical models.
bwd_estimate = fwd_full * 2.0
per_batch = fwd_full + bwd_estimate
epoch_sec = per_batch * n_train_batches

print(f"\n=== Epoch estimate ===")
print(f"Train batches per epoch: {n_train_batches}")
print(f"Per batch (forward):     {fwd_full:.2f}s")
print(f"Per batch (~backward):   {bwd_estimate:.2f}s  (rough 2x rule of thumb)")
print(f"Per batch (total):       {per_batch:.2f}s")
print(f"Per epoch:               {epoch_sec/60:.1f} min "
      f"({epoch_sec/3600:.2f} h)")
print(f"NUM_EPOCHS={NUM_EPOCHS} → total training: "
      f"{epoch_sec*NUM_EPOCHS/3600:.2f} h")

if _was_training:
    model.train()

del x_prof, x_full
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Train

In [ ]:
# define loss function and optimizer
if task == "multi-label, binary-class":
    criterion = nn.BCEWithLogitsLoss()
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9)

In [ ]:
# train

for epoch in range(NUM_EPOCHS):
    train_correct = 0
    train_total = 0
    test_correct = 0
    test_total = 0
    print('Epoch [%d/%d]'% (epoch+1, NUM_EPOCHS))
    model.train()
    for inputs, targets in tqdm(train_loader):
        inputs, targets = inputs.cuda(), targets.cuda()
        # forward + backward + optimize
        optimizer.zero_grad()
        outputs = model(inputs)

        if task == 'multi-label, binary-class':
            targets = targets.to(torch.float32)
            loss = criterion(outputs, targets)
        else:
            targets = targets.squeeze().long()
            loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

## Test

In [ ]:
# If the selected variant changes execution mode between train and test
# (e.g. quantum_softmax_gpu_qpu: trained on the analytic simulator, evaluated
# with finite shots), rebuild the model in eval mode and copy weights over.
_eval_swap = MODEL_BUILDERS[MODEL_VARIANT]["eval_swap"]
if _eval_swap is not None:
    _new_model = _eval_swap(n_classes).cuda()
    _new_model.load_state_dict(model.state_dict())
    model = _new_model
    print(f"Swapped model to eval mode for variant: {MODEL_VARIANT}")
else:
    print(f"No eval-mode swap needed for variant: {MODEL_VARIANT}")

In [ ]:
split = 'test'

model.eval()
y_true = torch.tensor([])
y_score = torch.tensor([])

data_loader = train_loader_at_eval if split == 'train' else test_loader

with torch.no_grad():
    for inputs, targets in data_loader:
        inputs = inputs.cuda()
        outputs = model(inputs)
        outputs = outputs.softmax(dim=-1)
        y_score = torch.cat((y_score, outputs.cpu()), 0)

    y_score = y_score.detach().numpy()

    evaluator = Evaluator(data_flag, split, size=224)
    metrics = evaluator.evaluate(y_score)

    print('%s  auc: %.3f  acc: %.3f' % (split, *metrics))

## Adversarial Robustness

reduce batch size for GPU limitation

In [ ]:
BATCH_SIZE = 5
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model.eval()

correct = 0
total = 0

atk = FGSM(model, eps=0.01)

for images, labels in test_loader:
    labels = labels.squeeze(1)
    images = atk(images, labels).cuda()
    outputs = model(images)

    _, predicted = torch.max(outputs.data, 1)

    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()

print('FGSM Robust accuracy: %.2f %%' % (100 * float(correct) / total))

In [ ]:
model.eval()

correct = 0
total = 0

atk = PGD(model, eps=8/255, alpha=4/255, steps=10, random_start=True)

for images, labels in test_loader:
    labels = labels.squeeze(1)
    images = atk(images, labels).cuda()
    outputs = model(images)

    _, predicted = torch.max(outputs.data, 1)

    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()

print('PGD Robust accuracy: %.2f %%' % (100 * float(correct) / total))